In [1]:
# Filename......: L13_OlisBahari_ITAI1371
# Language......: Python
# Tools.........: Visual Studio Code (VSC)
#               : Google Colab
# Class.........: ITAI 1371 Introduction to Machine Learning
# Semester......: Summer 2026
# Class Type....: Online
# Instructor....: Sitaram Ayyagari
# Student.......: Olis Bahari
# Version.......: V1.0
# Purpose.......: Compare manual data preprocessing with a Scikit-Learn
#                 pipeline using a Random Forest classifier on the Titanic dataset.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

In [3]:
# Load data
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Basic feature engineering and cleaning
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

X = df.drop(['Survived','Name', 'Ticket', 'PassengerId'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify feature types
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Manual Preprocessing
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_num = scaler.transform(X_test[numeric_features]) # Note: using .transform() here!

encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded_cat = encoder.fit_transform(X_train[categorical_features])
X_test_encoded_cat = encoder.transform(X_test[categorical_features])

# Combine preprocessed features
X_train_processed = np.hstack((X_train_scaled_num, X_train_encoded_cat.toarray()))
X_test_processed = np.hstack((X_test_scaled_num, X_test_encoded_cat.toarray()))

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)

print(f"Accuracy (Manual Method): {accuracy_score(y_test, y_pred):.2%}")

Accuracy (Manual Method): 82.68%


In [4]:
# Reload the data to start fresh
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a pipeline for numeric features
#    This pipeline will first impute missing 'Age' values with the median, then scale the features.
numeric_transformer = make_pipeline(
     SimpleImputer(strategy='median'),
     StandardScaler()
 )

# Create a pipeline for categorical features
#    This pipeline will first impute missing 'Embarked' values with the most frequent value, then one-hot encode.
categorical_transformer = make_pipeline(
     SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

# Use ColumnTransformer to apply different transformers to different columns
preprocessor = make_column_transformer(
     (numeric_transformer, ['Age', 'Fare', 'SibSp', 'Parch']),
     (categorical_transformer, ['Pclass', 'Sex', 'Embarked'])
 )

# Create the final, full pipeline
#    This chains the preprocessor and the final model together.
final_pipeline = make_pipeline(
     preprocessor,
     RandomForestClassifier(random_state=42)
)

# Fit and evaluate the entire pipeline in one step!
final_pipeline.fit(X_train, y_train)
y_pred_pipeline = final_pipeline.predict(X_test)

print(f"Accuracy (Pipeline Method): {accuracy_score(y_test, y_pred_pipeline):.2%}")

Accuracy (Pipeline Method): 82.68%


Reflective Knowledge Check

1. Code Comparison

Using a Pipeline helps keep your code simple, consistent, and less likely to have errors.
Combining preprocessing and model training in one workflow makes everything easier to manage.
This way, you can be sure that preprocessing is done the same way for both your training and test data.
It also helps you avoid mistakes, such as fitting a scaler or encoder on your test data.

Reflective Knowledge Check

2. Data Leakage Explained

Use scaler.fit_transform() only on your training data so the scaler learns the mean and standard deviation from that set. After that, apply scaler.transform() to your test data. Fitting the scaler on the test data can cause data leakage and make your model evaluation less trustworthy.

The Pipeline helps automate these steps. When you run final_pipeline.fit(X_train, y_train), it fits the preprocessing steps using just the training data. Later, when you use final_pipeline.predict(X_test), it applies the same preprocessing to the test data without fitting again.

Reflective Knowledge Check

 3. Extending the Pipeline

I suggest placing PCA after the preprocessor and before the RandomForestClassifier in the final pipeline. The steps would look like this:

Preprocessing → PCA → Random Forest

This approach first cleans, scales, and encodes the data. PCA then reduces the number of features before the processed data is sent to the Random Forest classifier.

Reflective Knowledge Check

4. Real-World Value

A single final_pipeline object is safer and more reliable because it manages the whole machine-learning workflow in the right order. The deployment team can enter raw data and get predictions without having to apply each preprocessing step and the model by hand.

If you use separate objects, you might skip steps, do them in the wrong order, or use different preprocessing settings by mistake. A single pipeline makes sure new data is handled the same way as the training data, which keeps deployment consistent and helps prevent errors.